# Phase 2: Data Preprocessing and Feature Engineering
## Customer Churn & LTV Prediction Engine

This notebook serves to prototype data cleaning pipelines, transformations, and feature engineering routines. Once verified, these routines are consolidated inside the corresponding package files `src/preprocessing.py` and `src/feature_engineering.py` for automated pipeline execution.

### Objectives:
1. Load dataset and apply cleaning methods (e.g. numeric castings and binary mappings).
2. Add engineered features (tenure cohorts, services density, and billing ratios).
3. Apply scaling, imputation, and OHE using Scikit-Learn's ColumnTransformer.
4. Partition features and serialize preprocessed dataset splits.

In [1]:
import sys
import os

# Ensure project root is in python path
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np

from src.data_loader import load_raw_data
from src.preprocessing import clean_data, get_preprocessor_pipeline, split_dataset
from src.feature_engineering import create_features
from src.utils import load_config

print("Preprocessing modules imported successfully!")

Preprocessing modules imported successfully!


In [2]:
# Load and clean
config = load_config()
try:
    df_raw = load_raw_data()
    df_clean = clean_data(df_raw)
    df_features = create_features(df_clean)
    print(f"Initial shape: {df_raw.shape} | Feature-engineered shape: {df_features.shape}")
except Exception as e:
    print(f"Could not run cleaning pipeline: {e}")

2026-06-10 21:20:19,084 - src.data_loader - INFO - Loading raw dataset from: ..\data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
2026-06-10 21:20:19,156 - src.data_loader - INFO - Successfully loaded dataset with shape: (7043, 21)
2026-06-10 21:20:19,159 - src.preprocessing - INFO - Cleaning and casting 'TotalCharges' to numeric float...
2026-06-10 21:20:19,203 - src.preprocessing - INFO - Imputing 11 missing TotalCharges for customers with 0 tenure to 0.0.
2026-06-10 21:20:19,208 - src.preprocessing - INFO - Mapping target column 'Churn' to binary (1/0)...
2026-06-10 21:20:19,212 - src.preprocessing - INFO - Data cleansing completed.
2026-06-10 21:20:19,213 - src.feature_engineering - INFO - Starting feature engineering pipeline...
2026-06-10 21:20:19,215 - src.feature_engineering - INFO - Adding feature: 'tenure_group' cohorts...
2026-06-10 21:20:19,226 - src.feature_engineering - INFO - Adding feature: 'total_services' count out of 6...
2026-06-10 21:20:19,239 - src.feature_engineering

Initial shape: (7043, 21) | Feature-engineered shape: (7043, 25)


In [1]:
# Splitting and Preprocessing Pipeline
if 'df_features' in locals():
    # Split
    X_train, X_test, y_train, y_test = split_dataset(df_features, target_col='Churn')
    
    # Preprocessor ColumnTransformer
    # Note: add custom feature engineering columns to config or handle manually
    preprocessor = get_preprocessor_pipeline()
    
    print("Fitting preprocessor pipeline on training features...")
    X_train_trans = preprocessor.fit_transform(X_train)
    X_test_trans = preprocessor.transform(X_test)
    
    print(f"Transformed training features shape: {X_train_trans.shape}")

In [3]:
# Load the raw dataset using the correct local path
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# 1. Cast TotalCharges to numeric, turning spaces (" ") into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Impute TotalCharges for tenure=0 to 0.0
# We find rows where tenure is 0 and fill their TotalCharges with 0.0
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0.0

# 3. Map Churn target to binary 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Check the results
print("Data types after cleaning:")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].dtypes)
print("\nNumber of missing values in TotalCharges:", df['TotalCharges'].isnull().sum())


Data types after cleaning:
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
Churn               int64
dtype: object

Number of missing values in TotalCharges: 0


In [6]:
# 1. Define the bin boundaries
bins = [0, 12, 24, 36, 48, 60, 72]

# 2. Define exactly 6 labels
labels = ["0-1 Year", "1-2 Years", "2-3 Years", "3-4 Years", "4-5 Years", "5-6 Years"]

# 3. Use pd.cut() and assign the output to df['tenure_group']
# We use include_lowest=True so that 0-tenure customers are included in the first bin.
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)

# Let's count how many customers fall into each group
print(df['tenure_group'].value_counts())


tenure_group
0-1 Year     2186
5-6 Years    1407
1-2 Years    1024
2-3 Years     832
4-5 Years     832
3-4 Years     762
Name: count, dtype: int64


In [7]:
# 1. List the 6 secondary services
services = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# 2. Check which columns equal 'Yes', then sum along the rows (axis=1)
# (True counts as 1, False counts as 0)
df['total_services'] = (df[services] == 'Yes').sum(axis=1)

# 3. Compare the average number of services for churned vs. non-churned customers
print("Average number of services by Churn status (0 = Stayed, 1 = Churned):")
print(df.groupby('Churn')['total_services'].mean())

# 4. View a sample of the results
print("\nSample of services and the engineered count:")
print(df[services + ['total_services']].head())


Average number of services by Churn status (0 = Stayed, 1 = Churned):
Churn
0    2.135292
1    1.768325
Name: total_services, dtype: float64

Sample of services and the engineered count:
  OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV  \
0             No          Yes               No          No          No   
1            Yes           No              Yes          No          No   
2            Yes          Yes               No          No          No   
3            Yes           No              Yes         Yes          No   
4             No           No               No          No          No   

  StreamingMovies  total_services  
0              No               1  
1              No               2  
2              No               2  
3              No               3  
4              No               0  


In [8]:
# 1. Calculate the Monthly to Total Charges ratio
# We use np.where to handle the edge case where TotalCharges is 0 to avoid dividing by zero.
df['charge_ratio'] = np.where(df['TotalCharges'] > 0, df['MonthlyCharges'] / df['TotalCharges'], 0.0)

# 2. Calculate Expected vs Actual Charges difference
# Expected total charges = tenure * MonthlyCharges
df['charges_difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])

# Let's inspect a few rows
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'charge_ratio', 'charges_difference']].head())


   tenure  MonthlyCharges  TotalCharges  charge_ratio  charges_difference
0       1           29.85         29.85      1.000000                0.00
1      34           56.95       1889.50      0.030140              -46.80
2       2           53.85        108.15      0.497920                0.45
3      45           42.30       1840.75      0.022980              -62.75
4       2           70.70        151.65      0.466205               10.25


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Separate features (X) and target (y)
# We drop customerID as it is just an identifier, and Churn because it's our target
X = df.drop(columns=['customerID', 'Churn'], errors='ignore')
y = df['Churn']

# 2. Perform train-test split with Stratification
# stratify=y ensures both train and test sets have the same ~26.5% churn rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Identify numerical and categorical columns
# (Including our new engineered features!)
numeric_cols = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 
    'total_services', 'charge_ratio', 'charges_difference'
]

categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 
    'PhoneService', 'MultipleLines', 'InternetService', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group'
]

# 4. Define Preprocessing Pipelines
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # In case of missing numbers
    ('scaler', StandardScaler())                  # Scale to mean=0, std=1
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # In case of missing categories
    ('onehot', OneHotEncoder(drop='first', sparse_output=False)) # OHE with dummy variable drop
])

# 5. Combine using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_cols),
        ('cat', cat_pipeline, categorical_cols)
    ],
    remainder='drop' # Drop any column not specified
)

# 6. Fit the preprocessor on training data and transform both sets
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Verify the shapes
print(f"Original training shape: {X_train.shape}")
print(f"Transformed training shape: {X_train_transformed.shape}")
print(f"Transformed testing shape: {X_test_transformed.shape}")


Original training shape: (5634, 23)
Transformed training shape: (5634, 38)
Transformed testing shape: (1409, 38)


In [10]:
import os
import joblib

# 1. Create directories if they don't exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 2. Save split datasets as CSVs (for record keeping/debugging)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# 3. Save the fitted preprocessor pipeline to disk
preprocessor_path = '../models/preprocessor.joblib'
joblib.dump(preprocessor, preprocessor_path)

print("Preprocessed splits and fitted pipeline successfully saved!")
print(f"Preprocessor saved to: {os.path.abspath(preprocessor_path)}")


Preprocessed splits and fitted pipeline successfully saved!
Preprocessor saved to: c:\Users\Rudra Pratap Giri\OneDrive\Desktop\Internship Project1\customer-churn-ltv-engine\models\preprocessor.joblib
